<a href="https://colab.research.google.com/github/yashjais2007/Gym_fitness_center_analysis/blob/main/Gym_fitness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#task 1: Load and inspect the membership dataset
import pandas as pd
import numpy as np

# Note: keep_default_na=False prevents pandas from turning the string "None" in Fitness_Class into NaN,
# while na_values=[""] ensures empty cells in Workout_Duration_Mins are still caught as missing.
df = pd.read_csv("12_gym_membership.csv", keep_default_na=False, na_values=[""])

print("--- Dataset Shape & Columns ---")
print(df.shape)
print(df.info())
print("\n--- First 5 Rows ---")
print(df.head())

--- Dataset Shape & Columns ---
(862, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 862 entries, 0 to 861
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Checkin_ID             862 non-null    object 
 1   Member_ID              862 non-null    object 
 2   Membership_Tier        862 non-null    object 
 3   Gender                 862 non-null    object 
 4   Checkin_Time           862 non-null    object 
 5   Workout_Duration_Mins  841 non-null    float64
 6   Fitness_Class          862 non-null    object 
 7   Monthly_Fee            862 non-null    int64  
dtypes: float64(1), int64(1), object(6)
memory usage: 54.0+ KB
None

--- First 5 Rows ---
  Checkin_ID Member_ID Membership_Tier  Gender         Checkin_Time  \
0     G00000     M0316        Platinum   Other  2026-05-01 22:00:00   
1     G00001     M0234           Basic    Male  2025-05-01 07:00:00   
2     G00002     M0094          

In [8]:
#task 2: Check for missing values and duplicate checkin_ID enteries
print("--- Missing Values per Column ---")
print(df.isnull().sum())

duplicate_count = df.duplicated(subset=["Checkin_ID"]).sum()
print(f"\nDuplicate Checkin_ID entries found: {duplicate_count}")


df = df.drop_duplicates(subset=["Checkin_ID"]).copy()
print(f"Rows after removing duplicates: {len(df)}")

--- Missing Values per Column ---
Checkin_ID                0
Member_ID                 0
Membership_Tier           0
Gender                    0
Checkin_Time              0
Workout_Duration_Mins    21
Fitness_Class             0
Monthly_Fee               0
dtype: int64

Duplicate Checkin_ID entries found: 12
Rows after removing duplicates: 850


In [9]:
#task 3: clean missing Workout_Duration_Mins values and explain the chosen method
median_duration = df["Workout_Duration_Mins"].median()
df["Workout_Duration_Mins"] = df["Workout_Duration_Mins"].fillna(median_duration)

print(f"Imputed missing Workout_Duration_Mins with median: {median_duration} mins")
print("Remaining missing values:", df["Workout_Duration_Mins"].isnull().sum())

Imputed missing Workout_Duration_Mins with median: 72.0 mins
Remaining missing values: 0


In [10]:
#task 4: convert Checkin_Time to datetime and extract weekday and hour of visit
df["Checkin_Time"] = pd.to_datetime(df["Checkin_Time"])
df["Weekday"] = df["Checkin_Time"].dt.day_name()
df["Hour"] = df["Checkin_Time"].dt.hour

print(df[["Checkin_ID", "Checkin_Time", "Weekday", "Hour"]].head(10))

  Checkin_ID        Checkin_Time    Weekday  Hour
0     G00000 2026-05-01 22:00:00     Friday    22
1     G00001 2025-05-01 07:00:00   Thursday     7
2     G00002 2026-10-14 17:00:00  Wednesday    17
3     G00003 2025-10-09 15:00:00   Thursday    15
4     G00004 2025-04-11 09:00:00     Friday     9
5     G00005 2025-02-17 20:00:00     Monday    20
6     G00006 2026-07-28 07:00:00    Tuesday     7
7     G00007 2025-03-17 21:00:00     Monday    21
8     G00008 2025-10-08 19:00:00  Wednesday    19
9     G00009 2026-03-24 17:00:00    Tuesday    17


In [11]:
#task 5: Calculate total valid check-ins and average workout duration
total_checkins = len(df)
avg_workout_duration = df["Workout_Duration_Mins"].mean()

print(f"Total Valid Check-ins: {total_checkins}")
print(f"Average Workout Duration: {avg_workout_duration:.2f} mins")

Total Valid Check-ins: 850
Average Workout Duration: 72.43 mins


In [13]:
#task 6: Compare visit frequency across membership tiers(e.g.,basic,gold,platinum)
tier_comparison = df.groupby("Membership_Tier").agg(
    Total_Visits=("Checkin_ID", "count"),
    Unique_Members=("Member_ID", "nunique"),
    Avg_Duration_Mins=("Workout_Duration_Mins", "mean")
)
tier_comparison["Avg_Visits_Per_Member"] = (
    tier_comparison["Total_Visits"] / tier_comparison["Unique_Members"]
).round(2)

print(tier_comparison)

                 Total_Visits  Unique_Members  Avg_Duration_Mins  \
Membership_Tier                                                    
Basic                     415             242          71.867470   
Gold                      293             212          74.440273   
Platinum                  142             122          69.950704   

                 Avg_Visits_Per_Member  
Membership_Tier                         
Basic                             1.71  
Gold                              1.38  
Platinum                          1.16  


In [14]:
# task 7: identify peak gym hours and busiest days of the week
hourly_visits = df["Hour"].value_counts().sort_values(ascending=False)
daily_visits = df["Weekday"].value_counts().sort_values(ascending=False)

peak_hour = hourly_visits.idxmax()
peak_hour_count = hourly_visits.max()

busiest_day = daily_visits.idxmax()
busiest_day_count = daily_visits.max()

print(f"Peak Gym Hour: {peak_hour}:00 ({peak_hour_count} check-ins)")
print(f"Busiest Day of the Week: {busiest_day} ({busiest_day_count} check-ins)")

print("\n--- Top 5 Peak Gym Hours ---")
print(hourly_visits.head(5))

print("\n--- All Weekdays Ranked by Visits ---")
print(daily_visits)

Peak Gym Hour: 7:00 (63 check-ins)
Busiest Day of the Week: Saturday (137 check-ins)

--- Top 5 Peak Gym Hours ---
Hour
7     63
13    63
21    51
18    51
5     46
Name: count, dtype: int64

--- All Weekdays Ranked by Visits ---
Weekday
Saturday     137
Monday       127
Wednesday    126
Thursday     125
Sunday       120
Tuesday      114
Friday       101
Name: count, dtype: int64


In [15]:
#task 8:rank fitness classes(e.g.,yoga,hiit,zumba)by total attendance
classes_only = df[df["Fitness_Class"] != "None"]

class_ranking = (
    classes_only["Fitness_Class"]
    .value_counts()
    .reset_index()
)
class_ranking.columns = ["Fitness_Class", "Total_Attendance"]
class_ranking["Rank"] = class_ranking["Total_Attendance"].rank(ascending=False, method="min").astype(int)

print(class_ranking)


  Fitness_Class  Total_Attendance  Rank
0      Strength               151     1
1          Spin               143     2
2         Zumba               136     3
3          HIIT               135     4
4          Yoga               132     5


In [16]:
#task 9: Compare average workout duration between male and female members
mf_df = df[df["Gender"].isin(["Male", "Female"])]

gender_comparison = mf_df.groupby("Gender")["Workout_Duration_Mins"].agg(
    Total_Visits="count",
    Mean_Duration="mean",
    Median_Duration="median"
).round(2)

print(gender_comparison)

        Total_Visits  Mean_Duration  Median_Duration
Gender                                              
Female           296          73.03             72.0
Male             292          73.10             72.0


In [17]:
#task 10: calculate monthly membership revenue by tier
unique_member_tiers = df.drop_duplicates(subset=["Member_ID", "Membership_Tier"])

monthly_revenue_by_tier = unique_member_tiers.groupby("Membership_Tier").agg(
    Active_Members=("Member_ID", "count"),
    Total_Monthly_Revenue=("Monthly_Fee", "sum"),
    Avg_Fee=("Monthly_Fee", "mean")
).round(2)

print(monthly_revenue_by_tier)

                 Active_Members  Total_Monthly_Revenue  Avg_Fee
Membership_Tier                                                
Basic                       242                 407258  1682.88
Gold                        212                 337288  1590.98
Platinum                    122                 213878  1753.10


In [18]:
# task 11: Members withfewer than 2 visits per month as low entanglement members
df["Year_Month"] = df["Checkin_Time"].dt.to_period("M")

monthly_member_visits = df.groupby(["Member_ID", "Year_Month"])["Checkin_ID"].count().reset_index(name="Visits_In_Month")

low_engagement = monthly_member_visits[monthly_member_visits["Visits_In_Month"] < 2]
unique_low_engagement_members = low_engagement["Member_ID"].unique()

print(f"Total Low-Engagement Member Count (< 2 visits/month): {len(unique_low_engagement_members)}")
print("Sample Low-Engagement Members:", unique_low_engagement_members[:15])

Total Low-Engagement Member Count (< 2 visits/month): 345
Sample Low-Engagement Members: ['M0001' 'M0002' 'M0003' 'M0004' 'M0005' 'M0006' 'M0007' 'M0008' 'M0009'
 'M0010' 'M0011' 'M0012' 'M0013' 'M0014' 'M0015']


In [20]:
#task 12: use numpy to calculate median,variance,and quartiles of workout duration
durations_arr = df["Workout_Duration_Mins"].to_numpy()

np_median = np.median(durations_arr)
np_variance = np.var(durations_arr, ddof=1)  # Sample variance (ddof=1)
q1, q2, q3 = np.percentile(durations_arr, [25, 50, 75])

print(f"Median Workout Duration: {np_median} mins")
print(f"Sample Variance: {np_variance:.2f}")
print(f"Q1 (25th Percentile): {q1} mins")
print(f"Q2 (50th Percentile / Median): {q2} mins")
print(f"Q3 (75th Percentile): {q3} mins")

Median Workout Duration: 72.0 mins
Sample Variance: 975.50
Q1 (25th Percentile): 47.0 mins
Q2 (50th Percentile / Median): 72.0 mins
Q3 (75th Percentile): 98.75 mins


In [21]:
#task 13: identify usually long or short workout sessions using the IQR method
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = df[
    (df["Workout_Duration_Mins"] < lower_bound) |
    (df["Workout_Duration_Mins"] > upper_bound)
]

print(f"IQR: {iqr} mins | Normal Range: [{lower_bound:.2f}, {upper_bound:.2f}] mins")
print(f"Number of Outlier Sessions: {len(outliers)}")
if not outliers.empty:
    print(outliers[["Checkin_ID", "Member_ID", "Workout_Duration_Mins"]])

IQR: 51.75 mins | Normal Range: [-30.62, 176.38] mins
Number of Outlier Sessions: 0


In [22]:
#task 14: create a workout intensity category based on duration: (short, moderate, extended)using documented thresholds
conditions = [
    df["Workout_Duration_Mins"] < 45,
    (df["Workout_Duration_Mins"] >= 45) & (df["Workout_Duration_Mins"] <= 90),
    df["Workout_Duration_Mins"] > 90
]
labels = ["Short", "Moderate", "Extended"]

df["Workout_Intensity"] = np.select(conditions, labels, default="Unknown")

print("--- Workout Intensity Breakdown ---")
print(df["Workout_Intensity"].value_counts())
print("\n", df[["Checkin_ID", "Workout_Duration_Mins", "Workout_Intensity"]].head(10))

--- Workout Intensity Breakdown ---
Workout_Intensity
Moderate    387
Extended    268
Short       195
Name: count, dtype: int64

   Checkin_ID  Workout_Duration_Mins Workout_Intensity
0     G00000                   30.0             Short
1     G00001                   71.0          Moderate
2     G00002                  129.0          Extended
3     G00003                   72.0          Moderate
4     G00004                  107.0          Extended
5     G00005                  100.0          Extended
6     G00006                   20.0             Short
7     G00007                  127.0          Extended
8     G00008                  100.0          Extended
9     G00009                  117.0          Extended


In [23]:
#task 15: write five facility management and retention recommendations
peak_hour = df["Hour"].value_counts().idxmax()
busiest_day = df["Weekday"].value_counts().idxmax()

top_class = df[df["Fitness_Class"] != "None"]["Fitness_Class"].value_counts().idxmax()
solo_visits_pct = (df["Fitness_Class"] == "None").mean() * 100

top_tier = df["Membership_Tier"].value_counts().idxmax()
low_eng_count = len(unique_low_engagement_members)

recommendations = [
    f"1. Optimize Peak-Hour Staffing: {busiest_day}s and {peak_hour}:00 see the highest check-in volume. Schedule extra floor trainers and pre-peak equipment checks ahead of this window to reduce wait times.",
    f"2. Expand Top-Performing Class Capacity: '{top_class}' is the highest-attended fitness class. Add extra {top_class} sessions during peak evening and morning slots on {busiest_day}s to capture unmet demand.",
    f"3. Convert Solo Gym-Goers to Group Classes: {solo_visits_pct:.1f}% of check-ins do not include a fitness class ('None'). Offer a complimentary trial class pass to solo visitors to boost routine consistency and retention.",
    f"4. Launch Automated Low-Engagement Outreach: {low_eng_count} members logged fewer than 2 visits in a month. Trigger automated check-in reminders and offer a free 1-on-1 trainer consultation to re-engage at-risk members.",
    f"5. Align Tier Pricing & Upgrade Pathways: '{top_tier}' generates the highest visit volume, while monthly fees (₹999, ₹1,499, ₹2,499) overlap across tiers. Standardize tier pricing and offer high-frequency ₹999/₹1,499 members a discounted 30-day Platinum upgrade trial."
]

print("--- Facility Management & Retention Recommendations ---")
for rec in recommendations:
    print(rec)

--- Facility Management & Retention Recommendations ---
1. Optimize Peak-Hour Staffing: Saturdays and 7:00 see the highest check-in volume. Schedule extra floor trainers and pre-peak equipment checks ahead of this window to reduce wait times.
2. Expand Top-Performing Class Capacity: 'Strength' is the highest-attended fitness class. Add extra Strength sessions during peak evening and morning slots on Saturdays to capture unmet demand.
3. Convert Solo Gym-Goers to Group Classes: 18.0% of check-ins do not include a fitness class ('None'). Offer a complimentary trial class pass to solo visitors to boost routine consistency and retention.
4. Launch Automated Low-Engagement Outreach: 345 members logged fewer than 2 visits in a month. Trigger automated check-in reminders and offer a free 1-on-1 trainer consultation to re-engage at-risk members.
5. Align Tier Pricing & Upgrade Pathways: 'Basic' generates the highest visit volume, while monthly fees (₹999, ₹1,499, ₹2,499) overlap across tiers. 